# Student dropout: explorationThis notebook is the exploration that chose the feature set and the model. It is notwhat produces the artifacts the web app serves - `train_model.py` is, and it is theonly thing that writes `model.joblib` and `scaler.joblib`.Run it top to bottom with the dependencies in `requirements-dev.txt`.

In [ ]:
import matplotlib.pyplot as pltimport pandas as pdimport seaborn as snsfrom sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import accuracy_score, confusion_matrix, f1_scorefrom sklearn.model_selection import train_test_splitfrom sklearn.preprocessing import StandardScaler

## The dataSeven features, already selected from the full UCI dataset, and the outcome.The outcome ships pre-encoded as 0/1/2, and **not** in alphabetical order, so themapping cannot be guessed from the integers. Class 1 is the largest group and hasby far the highest average units approved, which identifies it as Graduate. Gettingthis wrong silently swaps two classes in every chart and coefficient table below.

In [ ]:
raw_data = pd.read_csv('student_dropout.csv')X = raw_data.drop(columns=['Target'])y = raw_data['Target']classes = ['Dropout', 'Graduate', 'Enrolled']print(f'Data shape: {X.shape}')print(raw_data['Target'].value_counts().sort_index().rename(lambda i: classes[i]))

In [ ]:
# Sanity check on the encoding: graduates should approve the most units,# dropouts the fewest.raw_data.groupby('Target')[    ['Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)']].mean().rename(index=lambda i: classes[i]).round(2)

## Train and evaluateStratified split: Enrolled is only 18% of the rows, and an unstratified split leavesan unstable number of them in the test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.25, random_state=42, stratify=y)scaler = StandardScaler()X_train_scaled = scaler.fit_transform(X_train)X_test_scaled = scaler.transform(X_test)model = LogisticRegression(max_iter=1000, random_state=42)model.fit(X_train_scaled, y_train)y_pred = model.predict(X_test_scaled)print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')print(f'F1 (weighted): {f1_score(y_test, y_pred, average="weighted"):.4f}')

## Where the model failsThe confusion matrix is the interesting part. Dropout and Graduate separate cleanly.Enrolled does not: most of the students who are still enrolled get predicted as oneof the other two. That is the honest limit of this feature set, and it is why theheadline accuracy alone would overstate how useful the model is.

In [ ]:
cm = confusion_matrix(y_test, y_pred)plt.figure(figsize=(7, 5))sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',            xticklabels=classes, yticklabels=classes)plt.xlabel('Predicted')plt.ylabel('True')plt.title('Confusion matrix')plt.show()pd.DataFrame(cm, index=classes, columns=classes)

## What drives the predictionCoefficients are in standardised units, so they are comparable across features.Units approved dominates everything else: strongly positive for Graduate, stronglynegative for Dropout. Unemployment rate barely matters.

In [ ]:
coefficients = pd.DataFrame(model.coef_, index=classes, columns=X.columns).Tcoefficients.reindex(coefficients['Graduate'].abs().sort_values(ascending=False).index).round(3)